In [ ]:
import os
from huggingface_hub 
import pickle 
from transformers import pipeline
import torch.nn.functional as F
from torch import cuda, bfloat16
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

In [ ]:
cuda.is_available()

In [ ]:
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B", 
                                             resume_download = True,
                                             token=api_key,
                                             cache_dir = '/data_users1/sagar')
model = model.to('cuda')

In [ ]:
with open('../data/processed/winograd_pairs.pkl', 'rb') as f:
    sentences = pickle.load(f)

test_sent = sentences.get(10)
#test_sent['template'] = test_sent['template'].replace('BLANK', '[BLANK]')
test_sent

In [ ]:
sentence.split('[BLANK]')[0]

In [ ]:
sentence = test_sent['template']
options = ['her', 'his']

messages=[
        {
            "role": "system",
            "content":("Below you will find a passage in *bold* which contains precisely one instance of "
                       f"the token BLANK. You will also be provided two options. "
                       "Your task is to replace the BLANK token with one of the options provided. "
                       "The tasks are designed to be unambiguous, so please provide only one solution and "
                       "do not reorder the data.")
        },
        {
            "role": "user",
            "content": (f"Given this passage: *{sentence}*\n"
                        f"Replace the BLANK  with one of the following options: {options}")
        },
    {
        "role": "assistant",
        "content": sentence.split('BLANK')[0]
    }
    ]

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", token=api_key)

tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<<SYS>>\n{{ message['content'] }}\n<</SYS>>\n\n"
    "{% elif message['role'] == 'user' %}"
    "[INST] {{ message['content'] }} [/INST]"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] }}\n"
    "{% endif %}"
    "{% endfor %}"
)

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", continue_final_message=True).to("cuda:0")

outputs = model.generate(inputs, 
                         max_new_tokens = 6,
                         output_scores=True, 
                         return_dict_in_generate=True, 
                         output_hidden_states=True, 
                         do_sample = True, 
                         temperature = 0.5, 
                         pad_token_id=tokenizer.eos_token_id,
                        top_k=2)

#print(completion.choices[0].message)

In [ ]:
decoded_output = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)

In [ ]:
decoded_output

In [ ]:
input_length = inputs.shape[1]  # Number of tokens in the input prompt
generated_tokens = outputs.sequences[0][input_length:]  # Only new tokens

# Decode just the assistant's response
assistant_reply = tokenizer.decode(generated_tokens, skip_special_tokens=True)
assistant_reply

In [ ]:
# Step 3: Get scores (logits for each generated token)
scores = outputs.scores  # List of length = max_new_tokens

token_probs = []

for token_id, score in zip(generated_tokens, outputs.scores):
    if score.dim() == 2:
        score = score[0]  # from shape [1, vocab_size] to [vocab_size]
    
    probs = F.softmax(score, dim=-1)
    prob = probs[token_id].item()
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    token_probs.append((token, prob))

# Display
for token, prob in token_probs:
    print(f"{token}\t{prob:.4f}")
